# VM Resource Planner (Reactive, no forecast)

Notebook này chạy pipeline LP ở chế độ **reactive** cho **CẢ HAI scenarios**:

| Scenario | Objective | Mục tiêu |
|----------|-----------|----------|
| **OVERLOAD** | capacity | Minimize Resource Overload |
| **COST** | cost | Minimize Operational Cost |

## Output Files
- `lp_schedule_overload.csv` - LP schedule với objective=capacity
- `lp_schedule_cost.csv` - LP schedule với objective=cost
- `vm_resource_planning_reactive.json` - Combined metrics report

## Các bước chính
1. Load VM catalog (với `cost_per_hour`, `switching_cost`)
2. Stream `y_test` như đo đạc thời gian thực
3. Chuyển đo đạc → nhu cầu overflow CPU/RAM
4. Giải LP mỗi timestamp cho **cả 2 scenarios**
5. Lưu schedules + metrics

**Sau khi chạy notebook này, chạy `lp_vs_ppo_comparison.ipynb` để so sánh LP vs PPO.**



In [17]:
import json
import importlib
from pathlib import Path

import pandas as pd

# Reload module to ensure latest changes are loaded
import vm_resource_planner
importlib.reload(vm_resource_planner)

from vm_resource_planner import (
    load_vm_catalog,
    load_ground_truth_df,
    convert_forecasts_to_requirements,
    build_reactive_schedule,
    build_reactive_schedule_for_scenario,  # NEW: for both scenarios
    compute_lp_metrics,
    HOST_SPEC,
    VM_TYPES_FILE,
    RESULTS_DIR,
)

RESULTS_DIR.mkdir(exist_ok=True)
print("✓ Libraries & planner helpers loaded (reactive mode)")


✓ Libraries & planner helpers loaded (reactive mode)


In [18]:
vm_catalog = load_vm_catalog(VM_TYPES_FILE)

print("VM catalog (with switching_cost):")
for spec in vm_catalog:
    print(
        f"  • {spec['name']}: {spec['vcpus']} vCPUs, {spec['memory_gb']} GB, "
        f"${spec['cost_per_hour']}/h, switching=${spec['switching_cost']}"
    )

print(f"\nHost spec: {HOST_SPEC}")


VM catalog (with switching_cost):
  • B2s: 2 vCPUs, 4 GB, $0.0416/h, switching=$0.01
  • D2s_v3: 2 vCPUs, 8 GB, $0.096/h, switching=$0.02
  • D8s_v3: 8 vCPUs, 64 GB, $0.384/h, switching=$0.05
  • D32s_v3: 32 vCPUs, 128 GB, $1.536/h, switching=$0.1

Host spec: {'total_cpu_cores': 1, 'total_memory_gb': 4, 'cpu_threshold_pct': 70, 'memory_threshold_pct': 75}


In [19]:
ground_truth_df = load_ground_truth_df()
ground_truth_df.head()


,timestamp,memory_usage_pct,cpu_total_usage,system_load
0,2024-01-01 00:00:00,0.027578,-0.232042,0.527563
1,2024-01-01 00:00:30,0.027839,-0.485797,0.431887
2,2024-01-01 00:01:00,0.024400,-0.470571,0.001342
3,2024-01-01 00:01:30,0.019491,-0.501022,-0.237850
4,2024-01-01 00:02:00,0.031230,-0.419821,-0.142173


In [20]:
# Chuyển ground-truth đo đạc thành nhu cầu overflow
requirements_df = convert_forecasts_to_requirements(ground_truth_df, HOST_SPEC)
requirements_df[[
    'timestamp',
    'cpu_total_usage', 'cpu_required_cores', 'cpu_overflow_cores',
    'memory_usage_pct', 'memory_required_gb', 'memory_overflow_gb'
]].head()


,timestamp,cpu_total_usage,cpu_required_cores,cpu_overflow_cores,memory_usage_pct,memory_required_gb,memory_overflow_gb
0,2024-01-01 00:00:00,-0.232042,0.527563,0.0,0.027578,0.001103,0.0
1,2024-01-01 00:00:30,-0.485797,0.431887,0.0,0.027839,0.001114,0.0
2,2024-01-01 00:01:00,-0.470571,0.001342,0.0,0.024400,0.000976,0.0
3,2024-01-01 00:01:30,-0.501022,0.000000,0.0,0.019491,0.000780,0.0
4,2024-01-01 00:02:00,-0.419821,0.000000,0.0,0.031230,0.001249,0.0


In [21]:
# Build schedules cho CẢ HAI scenarios
print("Building LP schedules for BOTH scenarios...")

# Scenario 1: OVERLOAD (objective=capacity)
schedule_overload = build_reactive_schedule_for_scenario(requirements_df, vm_catalog, scenario="overload")
print(f"✓ LP Overload: {len(schedule_overload)} steps")

# Scenario 2: COST (objective=cost)
schedule_cost = build_reactive_schedule_for_scenario(requirements_df, vm_catalog, scenario="cost")
print(f"✓ LP Cost: {len(schedule_cost)} steps")

# Preview
print("\n=== LP Overload Schedule Preview ===")
display(schedule_overload[['timestamp', 'allocation', 'vm_total_count', 'vm_cost_per_hour', 'sla_violation']].head())


Building LP schedules for BOTH scenarios...
✓ LP Overload: 17150 steps
✓ LP Cost: 17150 steps

=== LP Overload Schedule Preview ===


,timestamp,allocation,vm_total_count,vm_cost_per_hour,sla_violation
0,2024-01-01 00:00:00,Host only,0,0.0,0
1,2024-01-01 00:00:30,Host only,0,0.0,0
2,2024-01-01 00:01:00,Host only,0,0.0,0
3,2024-01-01 00:01:30,Host only,0,0.0,0
4,2024-01-01 00:02:00,Host only,0,0.0,0


In [22]:
# Compute metrics cho cả hai scenarios
metrics_overload = compute_lp_metrics(schedule_overload)
metrics_cost = compute_lp_metrics(schedule_cost)

print("=== LP OVERLOAD Metrics ===")
for k, v in metrics_overload.items():
    print(f"  {k}: {v}")

print("\n=== LP COST Metrics ===")
for k, v in metrics_cost.items():
    print(f"  {k}: {v}")


=== LP OVERLOAD Metrics ===
  total_vm_cost: 4205.568
  total_switching_cost: 112.7
  total_cost: 4318.268
  total_cost_step: 148.6855666666667
  total_vms_sum: 2738
  avg_vms: 0.15965014577259476
  mean_cpu_utilization_pct: 0.8948371160095208
  mean_mem_utilization_pct: 0.0
  sla_violations: 0
  sla_violation_rate: 0.0

=== LP COST Metrics ===
  total_vm_cost: 167.02399999999997
  total_switching_cost: 24.73
  total_cost: 191.754
  total_cost_step: 26.32795
  total_vms_sum: 4015
  avg_vms: 0.23411078717201167
  mean_cpu_utilization_pct: 8.56099155045171
  mean_mem_utilization_pct: 0.0
  sla_violations: 0
  sla_violation_rate: 0.0


In [23]:
# Lưu kết quả cho CẢ HAI scenarios
# Paths
schedule_overload_path = RESULTS_DIR / "lp_schedule_overload.csv"
schedule_cost_path = RESULTS_DIR / "lp_schedule_cost.csv"
report_path = RESULTS_DIR / "vm_resource_planning_reactive.json"

# Save schedules
schedule_overload.to_csv(schedule_overload_path, index=False)
schedule_cost.to_csv(schedule_cost_path, index=False)

# Save combined JSON report
report_payload = {
    'model': 'vm_resource_planner_notebook_reactive',
    'host_spec': HOST_SPEC,
    'scenarios': {
        'overload': {
            'metrics': metrics_overload,
        },
        'cost': {
            'metrics': metrics_cost,
        }
    }
}

with open(report_path, 'w') as f:
    json.dump(report_payload, f, indent=2)

print(f"✓ Saved: {schedule_overload_path}")
print(f"✓ Saved: {schedule_cost_path}")
print(f"✓ Saved: {report_path}")


✓ Saved: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\lp_schedule_overload.csv
✓ Saved: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\lp_schedule_cost.csv
✓ Saved: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\vm_resource_planning_reactive.json


In [24]:
# Hoàn tất - Summary
print("="*65)
print("           LP REACTIVE COMPLETED (Both Scenarios)")
print("="*65)

print("\n📁 Output files:")
print(f"  • LP Overload: {schedule_overload_path}")
print(f"  • LP Cost:     {schedule_cost_path}")
print(f"  • JSON Report: {report_path}")

print("\n📊 Summary:")
print(f"  {'Metric':<30} {'OVERLOAD':>15} {'COST':>15}")
print("-"*60)
print(f"  {'Total VMs':<30} {metrics_overload['total_vms_sum']:>15} {metrics_cost['total_vms_sum']:>15}")
print(f"  {'VM Cost ($/h)':<30} {metrics_overload['total_vm_cost']:>15.4f} {metrics_cost['total_vm_cost']:>15.4f}")
print(f"  {'Switching Cost ($)':<30} {metrics_overload['total_switching_cost']:>15.4f} {metrics_cost['total_switching_cost']:>15.4f}")
print(f"  {'Total Cost ($/h + switch)':<30} {metrics_overload['total_cost']:>15.4f} {metrics_cost['total_cost']:>15.4f}")
print(f"  {'CPU Util %':<30} {metrics_overload['mean_cpu_utilization_pct']:>15.2f} {metrics_cost['mean_cpu_utilization_pct']:>15.2f}")
print(f"  {'SLA Violations':<30} {metrics_overload['sla_violations']:>15} {metrics_cost['sla_violations']:>15}")

# So sánh giữa 2 scenarios
if metrics_overload['total_cost'] > 0 and metrics_cost['total_cost'] > 0:
    diff_pct = (metrics_cost['total_cost'] - metrics_overload['total_cost']) / metrics_overload['total_cost'] * 100
    print(f"\n💡 COST scenario {'đắt hơn' if diff_pct > 0 else 'rẻ hơn'} OVERLOAD: {abs(diff_pct):.1f}%")

print("\n✓ Giờ có thể chạy lp_vs_ppo_comparison.ipynb để so sánh LP vs PPO!")


           LP REACTIVE COMPLETED (Both Scenarios)

📁 Output files:
  • LP Overload: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\lp_schedule_overload.csv
  • LP Cost:     E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\lp_schedule_cost.csv
  • JSON Report: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\vm_resource_planning_reactive.json

📊 Summary:
  Metric                                OVERLOAD            COST
------------------------------------------------------------
  Total VMs                                 2738            4015
  VM Cost ($/h)                        4205.5680        167.0240
  Switching Cost ($)                    112.7000         24.7300
  Total Cost ($/h + switch)            4318.2680        191.7540
  CPU Util %                                0.89            8.56
  SLA Violations                               